# LAB1 – Bài 2 (Trang 28): Khảo sát và tiền xử lý dữ liệu Bank marketing
**Sinh viên:** Đinh Văn Bình – **MSSV:** 6351071007

## Câu 6: Tiền xử lý các thuộc tính thiếu dữ liệu và xóa bỏ các thuộc tính không cần thiết

Như đã thấy ở Câu 2, dữ liệu Bank marketing ghi giá trị thiếu bằng chuỗi `'unknown'` (6 thuộc tính định tính), và `pdays = 999` nghĩa là “chưa từng liên hệ”.

In [1]:
print('Họ tên: Đinh Văn Bình - MSSV: 6351071007')

Họ tên: Đinh Văn Bình - MSSV: 6351071007


### Bước 1. Import thư viện và đọc dữ liệu, đổi `'unknown'` thành giá trị rỗng

In [2]:
import numpy as np
import pandas as pd

with open('bank-additional-full.csv') as f:
    df = pd.read_csv(f, sep=';')

df = df.replace('unknown', np.nan)
print('Kích thước ban đầu:', df.shape)
print('\nSố giá trị rỗng mỗi thuộc tính:')
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

Kích thước ban đầu: (41188, 21)

Số giá trị rỗng mỗi thuộc tính:
default      8597
education    1731
housing       990
loan          990
job           330
marital        80
dtype: int64


### Bước 2. Xóa các dòng trùng lặp
Dòng trùng không mang thêm thông tin và làm lệch thống kê.

In [3]:
print('Số dòng trùng lặp:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Kích thước sau khi xóa:', df.shape)

Số dòng trùng lặp: 12
Kích thước sau khi xóa: (41176, 21)


### Bước 3. Xóa các thuộc tính không cần thiết
- **default:** thiếu 20.9% (khoảng 8 600 dòng), phần còn lại gần như toàn bộ là `no` (chỉ có **3** dòng `yes`). Thuộc tính gần như hằng số nên không giúp phân biệt khách, vì vậy xóa bỏ (giống cách xóa `cabin`/`boat`/`body` ở Titanic).
- **duration:** chỉ biết được *sau* khi cuộc gọi kết thúc, lúc đó cũng đã biết kết quả `y` (nếu `duration = 0` thì chắc chắn `y = no`). Tài liệu gốc của bộ dữ liệu (UCI) khuyến nghị bỏ thuộc tính này khi xây dựng mô hình dự đoán thực tế.
- **pdays:** 96.3% giá trị là 999 (“chưa từng liên hệ”), về bản chất là thiếu dữ liệu. Thông tin này đã có trong `previous` và `poutcome`.

In [4]:
print('default:'); print(df['default'].value_counts(dropna=False), end='\n\n')
print('Tỷ lệ pdays = 999: %.1f%%' % ((df['pdays'] == 999).mean() * 100))
print('Số dòng duration = 0 và kết quả y:', df.loc[df['duration'] == 0, 'y'].value_counts().to_dict())

df = df.drop(['default', 'duration', 'pdays'], axis=1)
print('\nKích thước sau khi xóa thuộc tính:', df.shape)

default:
default
no     32577
NaN     8596
yes        3
Name: count, dtype: int64

Tỷ lệ pdays = 999: 96.3%
Số dòng duration = 0 và kết quả y: {'no': 4}

Kích thước sau khi xóa thuộc tính: (41176, 18)


### Bước 4. Điền giá trị thiếu cho các thuộc tính định tính còn lại
Các thuộc tính còn thiếu đều là định tính và thiếu ít (dưới 5%), nên điền bằng **giá trị xuất hiện nhiều nhất (mode)**, giống cách điền `embarked = 'S'` ở Titanic:
- `job` (0.8%), `marital` (0.19%), `education` (4.2%), `housing` (2.4%), `loan` (2.4%).

In [5]:
fill_cols = ['job', 'marital', 'education', 'housing', 'loan']
for col in fill_cols:
    mode = df[col].mode()[0]
    n = df[col].isnull().sum()
    df[col] = df[col].fillna(mode)
    print(f'{col:10s}: điền {n:4d} giá trị rỗng bằng {mode!r}')

job       : điền  330 giá trị rỗng bằng 'admin.'
marital   : điền   80 giá trị rỗng bằng 'married'
education : điền 1730 giá trị rỗng bằng 'university.degree'
housing   : điền  990 giá trị rỗng bằng 'yes'
loan      : điền  990 giá trị rỗng bằng 'no'


### Bước 5. Kiểm tra lại dữ liệu sau tiền xử lý

In [6]:
print('Kích thước sau tiền xử lý:', df.shape)
print('Tổng số giá trị rỗng còn lại:', df.isnull().sum().sum())
print()
df.info()

Kích thước sau tiền xử lý: (41176, 18)
Tổng số giá trị rỗng còn lại: 0

<class 'pandas.DataFrame'>
RangeIndex: 41176 entries, 0 to 41175
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41176 non-null  int64  
 1   job             41176 non-null  str    
 2   marital         41176 non-null  str    
 3   education       41176 non-null  str    
 4   housing         41176 non-null  str    
 5   loan            41176 non-null  str    
 6   contact         41176 non-null  str    
 7   month           41176 non-null  str    
 8   day_of_week     41176 non-null  str    
 9   campaign        41176 non-null  int64  
 10  previous        41176 non-null  int64  
 11  poutcome        41176 non-null  str    
 12  emp.var.rate    41176 non-null  float64
 13  cons.price.idx  41176 non-null  float64
 14  cons.conf.idx   41176 non-null  float64
 15  euribor3m       41176 non-null  float64
 16  nr.employed    

In [7]:
df.head(10)

,age,job,marital,education,housing,loan,contact,month,day_of_week,campaign,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,yes,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
5,45,services,married,basic.9y,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
6,59,admin.,married,professional.course,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
7,41,blue-collar,married,university.degree,no,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
8,24,technician,single,professional.course,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
9,25,services,single,high.school,yes,no,telephone,may,mon,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Nhận xét
- Ban đầu `isnull()` không phát hiện giá trị thiếu nào, vì dữ liệu dùng chuỗi `'unknown'` thay cho ô trống. Bước quan trọng nhất là đổi `'unknown'` thành `NaN` để nhận diện đúng **12 718** giá trị thiếu.
- Xóa **12 dòng trùng lặp**, còn lại **41 176** dòng.
- Xóa **3 thuộc tính không cần thiết:**
  - `default`: thiếu nhiều và gần như hằng số;
  - `duration`: gây rò rỉ thông tin (data leakage) cho mô hình dự đoán;
  - `pdays`: 96% là giá trị đặc biệt 999, trùng thông tin với `previous`/`poutcome`.
- Điền **mode** cho 5 thuộc tính định tính thiếu ít: `job → admin.`, `marital → married`, `education → university.degree`, `housing → yes`, `loan → no`. Vì tỷ lệ thiếu nhỏ nên cách điền này hầu như không làm thay đổi phân bố dữ liệu.
- Kết quả: dữ liệu còn **41 176 dòng × 18 cột** và **không còn giá trị rỗng**, sẵn sàng cho bước biến đổi và mô hình hóa tiếp theo.